In [1]:
using Lux, Reactant, Enzyme, MLUtils, NNlib
using Optimisers, Random, Statistics, Images
using LinearAlgebra, Images, JLD2, ComponentArrays
using Dates, Plots

In [2]:
model= Chain(
    Conv((17, 17), 2 => 256; pad = 8),
    SkipConnection(
        Chain(
            Conv((1, 1), 256 => 512, gelu),
            SkipConnection(
                Chain(
                    Conv((1, 1), 512 => 512, gelu),
                    Conv((1, 1), 512 => 512, gelu),
                    Conv((1, 1), 512 => 512, gelu),
                ),
                +
            ),
            Conv((1, 1), 512 => 256, gelu),
            Conv((1, 1), 256 => 256, gelu),
        ),
        +
    ),
    Conv((1, 1), 256 => 128, gelu),
    Conv((1, 1), 128 => 64, gelu),
    Conv((1, 1), 64 => 1)
)

Chain(
    layer_1 = Conv((17, 17), 2 => 256, pad=8),    # 148_224 parameters
    layer_2 = SkipConnection(
        connection = +,
        layers = Chain(
            layer_1 = Conv((1, 1), 256 => 512, gelu_tanh),  # 131_584 parameters
            layer_2 = SkipConnection(
                connection = +,
                layers = Chain(
                    layer_(1-3) = Conv((1, 1), 512 => 512, gelu_tanh),  # 787_968 (262_656 x 3) parameters
                ),
            ),
            layer_3 = Conv((1, 1), 512 => 256, gelu_tanh),  # 131_328 parameters
            layer_4 = Conv((1, 1), 256 => 256, gelu_tanh),  # 65_792 parameters
        ),
    ),
    layer_3 = Conv((1, 1), 256 => 128, gelu_tanh),  # 32_896 parameters
    layer_4 = Conv((1, 1), 128 => 64, gelu_tanh),  # 8_256 parameters
    layer_5 = Conv((1, 1), 64 => 1),              # 65 parameters
)         # Total: 1_306_113 parameters,
          #        plus 0 states.

In [ ]:
emb_dim = 1
load_model = false
include("compiler.jl")

train! (generic function with 1 method)

In [7]:
train!(data_trial, train_state)

(nothing, ConcretePJRTNumber{Float32, 1}(1.2167584f0), NamedTuple(), Lux.Training.TrainState{Lux.Training.TrainingBackendCache{Lux.Training.ReactantBackend{Static.False, Missing, Nothing, AutoEnzyme{Nothing, Nothing}}, Static.False, Nothing, @NamedTuple{compiled_grad_and_step_function::Reactant.Compiler.Thunk{typeof(ReactantExt.compute_gradients_internal_and_step!), Symbol("##compute_gradients_internal_and_step!_reactant#795329"), false, Tuple{GenericLossFunction{typeof(Lux.LossFunctionImpl.l2_distance_loss), typeof(mean)}, Chain{@NamedTuple{layer_1::Conv{typeof(identity), Int64, Int64, Tuple{Int64, Int64}, Tuple{Int64, Int64}, NTuple{4, Int64}, Tuple{Int64, Int64}, Int64, Nothing, Nothing, Static.True, Static.False}, layer_2::SkipConnection{Chain{@NamedTuple{layer_1::Conv{typeof(gelu_tanh), Int64, Int64, Tuple{Int64, Int64}, Tuple{Int64, Int64}, NTuple{4, Int64}, Tuple{Int64, Int64}, Int64, Nothing, Nothing, Static.True, Static.False}, layer_2::SkipConnection{Chain{@NamedTuple{layer_1

In [ ]:
function sinusoidal_embedding(t::Float32, embedding_dim::Int, max_positions::Int=10000)
    half_dim = embedding_dim ÷ 2
    emb_scale = log(Float32(max_positions)) / (half_dim - 1)
    emb = exp.(-emb_scale .* Float32.(0:half_dim-1))
    emb = t .* emb  # shape: (half_dim,)
    emb = vcat(sin.(emb), cos.(emb))  # shape: (embedding_dim,)
    return Float32.(emb)
end

In [ ]:
im_path = "../../../Images/64/"
# Hyperparameters for Variance Preserving (VP) SDE
const βmin = 0.1
const βmax = 20.0
T = 1


Xoshiro(0x68610a843b4dab22, 0x802c338967f8999f, 0xf921d4c1a4e84996, 0x4391427355a7d6d7, 0x9f879e100357c7ce)